In [1]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics.pairwise import cosine_similarity
from utility import generate_question
from utility import choose_question
from utility import retrain_on_subset
from utility import update_data 
from utility import get_best_match

In [2]:
df = pd.read_csv('onpiece.csv')
X = df.drop('Character', axis = 1)
names = df['Character']

In [4]:
import warnings
warnings.filterwarnings("ignore", category=UserWarning, module='sklearn')
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X,names)
importance = pd.Series(model.feature_importances_, index=X.columns)
importance.sort_values(ascending=False).to_csv('feature_importance.csv', header=True)

In [5]:
asked_questions = []
count = 0
user_responses = pd.Series(-1, index=X.columns)
while df.shape[0] > 5: 
    count += 1
    if (count % 5 == 0) and (df.shape[0] > 10):
        new_importance = retrain_on_subset(df, asked_questions)
        if new_importance is not None:
            importance = new_importance

    current_quest = choose_question(importance, asked_questions)
    
    if current_quest is None:
        break

    print(generate_question(current_quest))
    ans = input("(y/n)? ").lower()
    numeric_answer = 1 if ans == 'y' else 0
    user_responses[current_quest] = numeric_answer

    df = update_data(df, current_quest, numeric_answer)
    asked_questions.append(current_quest)

    X = df.drop('Character', axis=1)
    names = df['Character']


Is your character a Blood_X?
Is your character a Blood_S?
Is your character a Blood_F?
Is your character a height_tall?
Is your character a age_25_60?
Is your character a has_devil_fruit?


In [ ]:
venom = user_responses.values.reshape(1, -1)
print(get_best_match(df,venom))

TypeError: get_best_match() missing 1 required positional argument: 'user_answers_profile'